# 三维数据的可视化

## 本章要学的内容

三维数据没法直接用眼睛确认，所以 PyVista 里对数据做“切面”“描出同值的地方”“放上箭头”这些操作，再画成图。

* 用 `slice()` 取切面
* 用 `contour()` 做等值面
* 用箭头与流线表示向量场
* 用 `PolyData` 的布尔运算求两根圆管的交线

:::{note}
想交互地转起来，就用 `pyvistaqt.BackgroundPlotter()` 或 `add_mesh_slice()` 这样的部件，本书是 HTML，所以把同样的场景在后台画好、当作图输出。
:::

In [1]:
import vtk

import pyvista as pv

from helper.pyvista import plotter_to_iframe

try:  # 只在能用交互窗口的环境里
    import pyvistaqt as pvqt
except ImportError:
    pvqt = None

## 切面(切片)

看三维数据最简单的办法就是切片(切面)：用面切开，看那里出现的二维断面，三维的问题就变成了二维的问题，能移动面的位置与朝向，就有了往数据里窥看的感觉。

下面的代码用 `MultiBlockPlot3DReader()` 读数据文件，这个读取器用来读 PLOT3D 格式(流体计算里常用的格式)，用 `add_function()` 选要读入的标量与向量，参数的含义请看 VTK 的官方文档；读进来的结果是 `StructuredGrid`。

从上面的属性可以看出，`grid` 是形状 `(57, 33, 25)` 的格子；因为它是 `StructuredGrid`，各点的坐标在 `points` 里，虽然没有单元的数据，但点上带着 3 类数据，名字是 `"Density"`、`"Momentum"`、`"StagnationEnergy"`、`"VelocityMagnitude"`、`"Velocity"`。

- `"Density"` 是放在点的数据(`point_data`)里的密度。
- `"Velocity"` 是向量，也就是各点的速度。
- `"VelocityMagnitude"` 是速度的大小，上色时常用。

In [2]:
reader = pv.reader.MultiBlockPlot3DReader('data/combxyz.bin')
reader.add_q_files('data/combq.bin')
reader.add_function(100)
reader.add_function(153)
reader.add_function(200)
blocks = reader.read()
grid = blocks[0]
grid

StructuredGrid (0x1dfeabb4a60)
  N Cells:      43008
  N Points:     47025
  X Bounds:     0.000e+00, 1.651e+01
  Y Bounds:     -5.662e+00, 5.662e+00
  Z Bounds:     2.333e+01, 3.619e+01
  Dimensions:   57, 33, 25
  N Arrays:     6

`slice` 相关的过滤器从三维格子里取出二维的断面，常用的方法如下。

| 方法 | 做什么 |
|:--|:--|
| `slice()` | 决定法线切出 1 张 |
| `slice_orthogonal()` | 一次切出垂直于 X、Y、Z 的 3 张 |
| `slice_along_axis()` | 沿轴切 |
| `add_mesh_slice()` | 加上能交互移动的断面 |
| `add_mesh_slice_orthogonal()` | 加上能交互移动的正交断面 |

In [3]:
print(grid.point_data.active_scalars_name, grid.point_data.active_vectors_name)

Density Momentum


为了让图好看些，先做出格子外形的框：用 VTK 的类做框，再用 `pyvista.wrap()` 变成 PyVista 的 `PolyData`。

下面的代码输出外形的框、用 `slice_orthogonal()` 做出的 3 张正交断面，以及能自由移动面的部件。

部件需要交互窗口，所以在本地运行时用 `pyvistaqt`。

```python
import pyvistaqt as pvqt

plotter = pvqt.BackgroundPlotter()
plotter.add_mesh(outline)
plotter.add_mesh(grid.slice_orthogonal(), cmap="cool")
plotter.add_mesh_slice(grid, cmap="cool")   # 这个面可以拖动
plotter.show()
```

本书则把同样的场景在后台画好、当作图输出。

In [4]:
sgof = vtk.vtkStructuredGridOutlineFilter()
sgof.SetInputData(grid)
sgof.Update()
outline = pv.wrap(sgof.GetOutput())

下面的代码显示外形的框、`grid.slice_orthogonal()` 得到的正交切片，以及能自由调整切面的部件。因为这里含有交互用的部件，所以没法直接在 Jupyter Notebook 里运行，于是用 `pyvistaqt.BackgroundPlotter()` 在另一个窗口里绘制。

![](images/pyvista-slice.png)

In [5]:
# 把 3 张正交断面与 x 为定值的断面叠起来
plotter = pv.Plotter()
plotter.add_mesh(outline)
plotter.add_mesh(grid.slice_orthogonal(), cmap="cool")
plotter.add_mesh(grid.slice(normal="x"), cmap="cool", opacity=0.6)
plotter_to_iframe(plotter)

C:\Users\ruoyu\Documents\notebooks\scipybook\scipybook_cn\helper\pyvista.py:12: PyVistaDeprecationWarning: Plotter.export_vtksz is deprecated. Use plotter.trame.export_vtksz(...) instead. Install via `pip install trame-pyvista`.
  zip_data = plotter.export_vtksz(filename=None)


## 等值面

等值面是把标量场里同值的点连起来的曲面，可以想成地图上等高线的三维版。

PyVista 里用 `contour()` 计算等值面：`isosurfaces` 是做几张面、`scalars` 是用哪个标量数组。

下面的程序用上一节的流体数据输出等值面。

In [6]:
plotter = pv.Plotter()
plotter.add_mesh(outline)
plotter.add_mesh(grid.contour(isosurfaces=8, scalars="Density"), cmap='cool', opacity=0.3)
plotter_to_iframe(plotter)

C:\Users\ruoyu\Documents\notebooks\scipybook\scipybook_cn\helper\pyvista.py:12: PyVistaDeprecationWarning: Plotter.export_vtksz is deprecated. Use plotter.trame.export_vtksz(...) instead. Install via `pip install trame-pyvista`.
  zip_data = plotter.export_vtksz(filename=None)


In [7]:
# 只输出 1 张等值面，值取密度的平均。
density = grid["Density"]
level = float((density.min() + density.max()) / 2)

plotter = pv.Plotter()
plotter.add_mesh(outline)
plotter.add_mesh(
    grid.contour(isosurfaces=[level], scalars="Density"), cmap="cool"
)
plotter_to_iframe(plotter)

上面的例子里，一张等值面上哪里的颜色都一样，因为面上的标量值(这里是密度)相同，上色也用了那个值。

有时想用别的值来上色，下面的程序就是换用另一个标量的办法。

给 `contour()` 的 `isosurfaces` 传列表，就会按那些值各做一张面；用 `add_mesh()` 的 `scalars` 选上色用的数组。

:::{note}
用 `add_mesh_isovalue()` 能一边拖滑块一边看等值面，这也需要交互窗口。
:::

In [8]:
plotter = pv.Plotter()
plotter.add_mesh(outline)
plotter.add_mesh(grid.contour(isosurfaces=[0.3], scalars="Density"), cmap='cool', scalars="VelocityMagnitude")
plotter_to_iframe(plotter)

## 向量场

前面的例子里用断面与等值面看了密度的分布：密度在每点是一个数(标量)，所以能表示成标量场。

而各点的速度是带方向与大小的向量，所以速度的分布是向量场，这里用箭头与流线来表示。

In [9]:
import vtk
mp = vtk.vtkMaskPoints() #{1}
mp.SetRandomMode(True)
mp.SetOnRatio(100)
mp.SetInputData(grid)
mp.Update()
rand_points = pv.wrap(mp.GetOutput())

sphere = pv.Sphere(radius=2, center=grid.center, phi_resolution=8, theta_resolution=8)
lines = grid.streamlines_from_source(sphere, integration_direction='forward')
tubes = lines.tube(radius=0.05)

plotter = pv.Plotter()
plotter.add_mesh(outline)
plotter.add_mesh(rand_points.glyph(factor=2.0), scalars='VelocityMagnitude')
plotter.add_mesh(sphere, style='wireframe')
plotter.add_mesh(tubes, scalars='VelocityMagnitude')
plotter_to_iframe(plotter)

In [10]:
# 以球为出发点的流线(位置固定)
sphere = pv.Sphere(
    radius=2, center=grid.center, phi_resolution=8, theta_resolution=8
)
lines = grid.streamlines_from_source(sphere, integration_direction="forward")
tubes = lines.tube(radius=0.05)

plotter = pv.Plotter()
plotter.add_mesh(outline)
plotter.add_mesh(sphere, style="wireframe")
plotter.add_mesh(tubes, scalars="VelocityMagnitude")
plotter_to_iframe(plotter)

下面的代码做出含 4 个要素的图：外形的框、稀疏放置的速度箭头、当作流线出发点的球，以及流线。

用 `add_sphere_widget()` 能拖动球来移动流线的出发点，要让它重算流线，就传进 `callback`。

```python
class StreamPlot(pvqt.BackgroundPlotter):
    def __init__(self, grid):
        super().__init__()
        self.grid = grid
        self.stream_actor = None
        self.add_mesh(outline)
        self.add_sphere_widget(
            self.callback, center=grid.center, radius=2,
            theta_resolution=8, phi_resolution=8,
            style="wireframe", pass_widget=True,
        )

    def callback(self, center, widget):
        # 球的位置一变，就把流线重做一遍再换上
        sphere = pv.PolyData()
        widget.GetPolyData(sphere)
        tubes = self.grid.streamlines_from_source(
            sphere, integration_direction="forward"
        ).tube(radius=0.05)
        if self.stream_actor is None:
            self.stream_actor = self.add_mesh(tubes, scalars="VelocityMagnitude")
        else:
            arr = tubes["VelocityMagnitude"]
            self.stream_actor.mapper.dataset = tubes
            self.stream_actor.mapper.scalar_range = arr.min(), arr.max()
```

本书把球固定在一个位置来输出流线。

In [11]:
from dataclasses import dataclass

@dataclass
class Tube:
    inner: pv.PolyData
    outer: pv.PolyData
    tube: pv.PolyData

def make_tube(height, radius, resolution, direction):
    cs1 = pv.Cylinder(height=height, radius=radius[0], resolution=resolution, direction=direction)
    cs2 = pv.Cylinder(height=height+0.1, radius=radius[1], resolution=resolution, direction=direction)

    cs1 = cs1.triangulate().subdivide(nsub=1, subfilter='loop')
    cs2 = cs2.triangulate().subdivide(nsub=1, subfilter='loop')
    tube = cs1.boolean_difference(cs2)
    return Tube(outer=cs1, inner=cs2, tube=tube)

tube1 = make_tube(5, [1, 0.8], 32, (0, 0, 1))
tube2 = make_tube(5, [0.7, 0.55], 32, (0, 1, 0))

## 两根圆管的交线

互相垂直的两根圆柱相交时会出现 8 条交线，这里用 `PolyData` 的布尔运算做出两根圆管，再求交线。

`make_tube()` 用 `Cylinder()` 做出内侧与外侧的圆柱，用 `triangulate()` 把所有面变成三角形，再用 `subdivide()` 细分。

然后用 `boolean_difference()` 从外侧减去内侧，就得到圆管。

In [12]:
tube1_with_hole = tube1.tube.boolean_difference(tube2.inner)
tube2_with_hole = tube2.tube.boolean_difference(tube1.inner)
intersection = tube1.tube.boolean_intersection(tube2.tube)

接着处理两根圆管重叠的地方。

从 `tube1` 减去 `tube2` 的内侧，就得到 `tube2` 插进去、带孔的 `tube1_with_hole`；用 `boolean_intersection()` 则只留下两根重叠的部分。

In [13]:
plotter = pv.Plotter()
plotter.add_mesh(tube1_with_hole, color='gray', opacity=0.5)
plotter.add_mesh(tube2_with_hole, color='gray', opacity=0.5)
plotter.add_mesh(intersection, color='red')
plotter_to_iframe(plotter)

把三个部分画成图。

In [14]:
edges = intersection.extract_feature_edges().clean(tolerance=1e-2, lines_to_points=False).connectivity()
edges

PolyData (0x1dff3e67a00)
  N Cells:    2472
  N Points:   1237
  N Strips:   0
  X Bounds:   -6.966e-01, 6.966e-01
  Y Bounds:   -9.952e-01, 9.952e-01
  Z Bounds:   -6.966e-01, 6.966e-01
  N Arrays:   10

交线(重叠的边缘)用 `extract_feature_edges()` 取出：它取网格上尖锐的边缘，用 `feature_angle` 以度为单位决定尖锐的阈值，这里默认的 30 度足够。

用 `clean()` 把靠得近的点合并起来：设 `lines_to_points=False`，可以避免变短的线被换成点。

再用 `connectivity()` 给连在一起的部分编上号(`RegionId`)。

In [15]:
plotter = pv.Plotter()
plotter.add_mesh(edges, cmap='tab10', scalars='RegionId')
plotter_to_iframe(plotter)

C:\Users\ruoyu\Documents\notebooks\scipybook\scipybook_cn\helper\pyvista.py:12: PyVistaDeprecationWarning: Plotter.export_vtksz is deprecated. Use plotter.trame.export_vtksz(...) instead. Install via `pip install trame-pyvista`.
  zip_data = plotter.export_vtksz(filename=None)


接着按 `RegionId` 的值分开颜色，输出 `edges`。

In [16]:
edge = edges.threshold((-0.1, 1.1), scalars='RegionId')
edge

UnstructuredGrid (0x1dff4230880)
  N Cells:    690
  N Points:   345
  X Bounds:   -6.966e-01, 6.966e-01
  Y Bounds:   -7.962e-01, 7.962e-01
  Z Bounds:   -6.966e-01, 6.966e-01
  N Arrays:   10

`threshold()` 只取出 `scalars` 的值落在范围内的单元，结果是 `UnstructuredGrid`，单元与点的关系在 `cells` 里；看 `N Cells` 与 `N Points` 就知道同一条边缘被重复放进去了。

In [17]:
from collections import defaultdict

from matplotlib.cm import tab10

def connect_lines(lines):
    edges = defaultdict(set)    
    for _, s, e in lines.reshape(-1, 3).tolist():
        edges[s].add(e)
        edges[e].add(s)

    while True:
        if not edges:
            break
        poly = [next(iter(edges.keys()))]
        while True:
            e = poly[-1]
            neighbours = edges[e]
            if not neighbours:
                break
            n = neighbours.pop()
            try:
                edges[n].remove(e)
            except:
                pass
            poly.append(n)
        yield poly
        edges = {k:v for k,v in edges.items() if v}

plotter = pv.Plotter()
for color, v in zip(tab10.colors, set(edges['RegionId'])):
    edge = edges.threshold((v - 0.1, v + 0.1), scalars='RegionId')
    loop = sorted(connect_lines(edge.cells), key=len)[-1]
    points = edge.points[loop]
    plotter.add_points(points, color='gray', opacity=0.5)
    plotter.add_mesh(pv.lines_from_points(points), line_width=2, color=color)
plotter_to_iframe(plotter)

## 小结

* 三维数据靠 3 种看法来理解：用面切开、描出同值的地方、放上箭头。
* 用 `slice()` 取断面，用 `slice_orthogonal()` 取 3 张正交断面。
* 用 `contour()` 做等值面，`isosurfaces` 是面的个数、`scalars` 是用的数组。
* 向量场用 `glyph()` 的箭头与 `streamlines_from_source()` 的流线表示。
* 用 `PolyData` 的布尔运算(`boolean_difference`、`boolean_intersection`)能做出相交的形状。
* 交线用 `extract_feature_edges()` 与 `connectivity()` 取出，再按 `RegionId` 分色。
* 交互部件(`add_mesh_slice`、`add_sphere_widget`)在 `pyvistaqt` 的窗口里用。